# Day 1 - letters + INCLUDE keypoints (CPU only, Internet ON)
Inputs needed: `slr-code`, ASL Alphabet (https://www.kaggle.com/datasets/grassknoted/asl-alphabet) and, optionally, an ISL alphabet image dataset. The pretrained-ASL smoke test and the `asl-signs` competition are NOT needed here: they belong to Day 2 (ASL).

In [ ]:
# Finds the uploaded "slr-code" dataset wherever Kaggle mounts it (the path differs between Kaggle versions).
import glob, os, shutil, subprocess, sys, zipfile
ON_KAGGLE = os.path.exists("/kaggle/input")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
CODE = f"{WORK}/code"
hits = [h for d in ("*", "*/*", "*/*/*") for h in glob.glob(f"/kaggle/input/{d}/slr/__init__.py")] + glob.glob("/content/slr_code/**/slr/__init__.py", recursive=True)  # shallow search: ASL Alphabet has ~87k files
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE, dirs_exist_ok=True)
else:
    zips = [h for d in ("*", "*/*", "*/*/*") for h in glob.glob(f"/kaggle/input/{d}/slr_code.zip")] + glob.glob("/content/**/slr_code.zip", recursive=True)
    if zips:
        zipfile.ZipFile(zips[0]).extractall(CODE)
    else:
        print("slr-code NOT FOUND. Add it: right sidebar -> + Add Input -> Your Datasets -> slr-code. Currently attached:")
        for p in sorted(glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*")):
            print("  ", p)
        raise SystemExit("Attach the slr-code dataset, then run this cell again.")
os.chdir(CODE)
print("code ready in", CODE, os.listdir(CODE))
# Install each package on its own so one failure cannot skip the others. Kaggle now runs Python 3.13, so we use the
# current mediapipe (Tasks API) - not the old pinned version.
for pkg in ("mediapipe", "remotezip", "opencv-python-headless"):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
    print(pkg, "OK" if r.returncode == 0 else "FAILED\n" + r.stderr[-600:])
import mediapipe; print("mediapipe", mediapipe.__version__)

## 1. Letters: ~300 images per letter -> hand landmarks -> classifier (minutes on CPU)
Open the *Input* panel (right sidebar), hover the dataset, click the copy-path icon, paste below. Set `ISL_IMAGES = None` if you have no ISL letter images yet - ASL still runs, ISL letters can be added later.

In [ ]:
import glob
_asl = [p for d in ("*", "*/*", "*/*/*") for p in glob.glob(f"/kaggle/input/{d}") if "asl-alphabet" in p.lower() and os.path.isdir(p)]
print("ASL alphabet candidates:", _asl)
ASL_IMAGES = min(_asl, key=len)                      # shortest = the dataset root; class folders are auto-detected
ISL_IMAGES = None                                    # e.g. "/kaggle/input/<your-isl-alphabet-dataset>"
for lang, root in [("asl", ASL_IMAGES), ("isl", ISL_IMAGES)]:
    if root is None:
        print("skipping", lang); continue
    !python scripts/letters.py extract --images $root --lang $lang --out $WORK/letters --per-class 300 --workers 4
    !python scripts/letters.py train --lang $lang --data $WORK/letters --out $WORK/models

## 2. INCLUDE-50 keypoints (reads only the 50 signs' videos from each Zenodo part, deletes each video right after)
Run `--list` first to check Internet/Zenodo. The second cell is the long one - use Save & Run All for it.

In [ ]:
!python scripts/include_extract.py --out $WORK/include50 --list

In [ ]:
!python scripts/include_extract.py --out $WORK/include50 --workers 8 --stride 2
!du -sh $WORK/include50

## 3. Save everything
Save Version -> Save & Run All, then Output -> **New Dataset** to keep `models/`, `letters/`, `include50/`.